# Project 1 — Object-Oriented Library Lending System

Build a fully testable in-memory library with registered members, books, and active loans. Practice dataclasses, domain modeling, composition, validation, unit testing, and JSON persistence.

**Level:** Intermediate project · **Suggested time:** 3–5 hours · **Prerequisites:** Intermediate lessons 00–19.

## Project requirements

- Model an immutable Book record
- Implement borrow and return rules in a library service
- Reject duplicate and invalid operations
- Save and load state using JSON and temporary paths
- Prove behavior with unittest

**Instructions:** Study the explanations and runnable implementation, then complete the extension challenges. All examples use the Python standard library and offline data. Run the notebook from a fresh kernel to test reproducibility.

## 1. Domain records and invariants

Use a frozen dataclass for identifying books; validate ids and titles. Domain invariants belong near object construction so every subsequent operation can rely on them.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Book:
    book_id: str
    title: str

    def __post_init__(self):
        if not isinstance(self.book_id, str) or not self.book_id.strip():
            raise ValueError("book id required")
        if not isinstance(self.title, str) or not self.title.strip():
            raise ValueError("title required")

print(Book("B001", "Python Handbook"))

**Design question:** Why is a title not necessarily a unique identifier?

## 2. Service object and state ownership

A service composes books, members, and loans. The dictionaries and sets are created separately for each instance so data does not leak between independent libraries. Each action checks relevant invariants before modifying state.

In [ ]:
class Library:
    def __init__(self):
        self.books = {}
        self.members = set()
        self.loans = {}

    def add_book(self, book):
        if book.book_id in self.books:
            raise ValueError("duplicate book")
        self.books[book.book_id] = book

    def register(self, member_id):
        if not isinstance(member_id, str) or not member_id.strip():
            raise ValueError("invalid member")
        self.members.add(member_id)

    def borrow(self, book_id, member_id):
        if book_id not in self.books:
            raise KeyError("unknown book")
        if member_id not in self.members:
            raise KeyError("unknown member")
        if book_id in self.loans:
            raise ValueError("already borrowed")
        self.loans[book_id] = member_id

    def return_book(self, book_id, member_id):
        if self.loans.get(book_id) != member_id:
            raise ValueError("no matching loan")
        del self.loans[book_id]

    def available(self):
        return [book for key, book in sorted(self.books.items()) if key not in self.loans]

library = Library()
library.add_book(Book("B1", "Learning Python"))
library.add_book(Book("B2", "Practical Data"))
library.register("M1")
library.borrow("B1", "M1")
print(library.available())

**Design question:** Would marking the Book object itself as 'borrowed' make the object model simpler or more fragile?

## 3. Explicit tests

Tests create fresh libraries to avoid state leaking between cases. Test normal borrowing, double loans, and unknown members using the public interface rather than reading internal implementation details.

In [ ]:
import unittest

class LibraryTests(unittest.TestCase):
    def setUp(self):
        self.lib = Library()
        self.lib.add_book(Book("A", "A book"))
        self.lib.register("member")
    def test_borrow_return(self):
        self.lib.borrow("A", "member")
        self.assertEqual(self.lib.available(), [])
        self.lib.return_book("A", "member")
        self.assertEqual(len(self.lib.available()), 1)
    def test_double_loan(self):
        self.lib.borrow("A", "member")
        with self.assertRaises(ValueError):
            self.lib.borrow("A", "member")
    def test_unregistered(self):
        with self.assertRaises(KeyError):
            self.lib.borrow("A", "unknown")

assert unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(LibraryTests)
).wasSuccessful()

**Design question:** Which invalid-return cases are not yet covered by these tests?

## 4. JSON round trip

Convert records to simple dictionaries before serialization. Rebuild the service by calling its validated methods rather than assigning untrusted fields directly. TemporaryDirectory makes the example safe to rerun.

In [ ]:
import json
from dataclasses import asdict
from pathlib import Path
from tempfile import TemporaryDirectory

def export_library(service):
    return json.dumps({
        "books": [asdict(book) for book in service.books.values()],
        "members": sorted(service.members),
        "loans": service.loans,
    }, indent=2)

def import_library(text):
    payload = json.loads(text)
    result = Library()
    for raw in payload["books"]:
        result.add_book(Book(raw["book_id"], raw["title"]))
    for member in payload["members"]:
        result.register(member)
    for book_id, member_id in payload["loans"].items():
        result.borrow(book_id, member_id)
    return result

with TemporaryDirectory() as folder:
    path = Path(folder) / "library.json"
    path.write_text(export_library(library), encoding="utf-8")
    loaded = import_library(path.read_text(encoding="utf-8"))
    assert loaded.loans == library.loans
    print("Restored:", loaded.loans)

**Design question:** What errors should a loader report if a JSON file has missing or invalid fields?

## Independent extension challenges

Add features in small increments. For each change, decide how to validate input and how to test failures.

**Challenge 1:** Support multiple physical copies of the same book using distinct book IDs.

In [ ]:
# TODO: Write code and assertions for challenge 1.

**Challenge 2:** Add due dates and an overdue-loan report using datetime.date.

In [ ]:
# TODO: Write code and assertions for challenge 2.

**Challenge 3:** Implement a reservation queue with a documented priority rule.

In [ ]:
# TODO: Write code and assertions for challenge 3.

**Challenge 4:** Extract models and service functions into modules and create an argparse CLI.

In [ ]:
# TODO: Write code and assertions for challenge 4.

## Portfolio quality checklist

- [ ] The implementation separates domain logic from I/O.
- [ ] There are tests for both normal and invalid inputs.
- [ ] Data validation is explicit and error messages are useful.
- [ ] Resources are closed and examples are safe to rerun.
- [ ] No live API calls or credentials are necessary to test.
- [ ] A README explains the project design, assumptions, and usage.